In [12]:
from pathlib import Path
import sys

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Current folder:", Path.cwd())
print("Project folder:", project_root)
print("Does src exist:", (project_root / "src").exists())

Current folder: d:\Data Scientist\ExamUppgift\ecommerce-team2\ecommerce-team2\notebooks
Project folder: d:\Data Scientist\ExamUppgift\ecommerce-team2\ecommerce-team2
Does src exist: True


In [25]:
import pandas as pd

from src.metrics import (
    revenue_by_group,
    revenue_by_month,
    total_revenue,
    total_units
)

from src.viz import plot_monthly_revenue

In [26]:
file_path = project_root / "data" / "ecommerce_sales.csv"

df = pd.read_csv(file_path)

df.columns = df.columns.str.strip()

df["date"] = pd.to_datetime(df["date"], errors="coerce")

display(df.head())

print("Antal rader och kolumner:", df.shape)

,order_id,date,city,category,price,units,revenue
0,100000,2024-04-19,Stockholm,Sports,530.95,4,2123.80
1,100001,2024-05-06,Malmö,Sports,653.28,3,1959.84
2,100002,2024-03-07,Uppsala,Clothing,177.98,4,711.92
3,100003,2024-04-08,Göteborg,Home,348.57,3,1045.71
4,100004,2024-01-18,Uppsala,Electronics,777.07,3,2331.21


Antal rader och kolumner: (2500, 7)


In [27]:
revenue_by_category = revenue_by_group(df, "category")
revenue_by_city = revenue_by_group(df, "city")

display(revenue_by_category)
display(revenue_by_city)


,category,total_revenue
0,Electronics,1364228.07
1,Sports,734653.42
2,Clothing,586744.91
3,Home,511070.87
4,Beauty,163053.97
5,Toys,143015.99


,city,total_revenue
0,Stockholm,1227201.94
1,Göteborg,859019.65
2,Malmö,567926.32
3,Uppsala,551478.42
4,Västerås,297140.90


Electronics står för högst intäkt bland kategorierna, följt av Sports och Clothing. Stockholm är staden med högst total intäkt, följt av Göteborg och Malmö.  

In [28]:
kategori_analys = df.groupby("category").agg(
    total_intäkt=("revenue", "sum"),
    antal_sålda_enheter=("units", "sum"),
    genomsnittligt_pris=("price", "mean")
).reset_index()

kategori_analys = kategori_analys.sort_values(
    by="total_intäkt",
    ascending=False
)

kategori_analys = kategori_analys.reset_index(drop=True)

display(kategori_analys)

,category,total_intäkt,antal_sålda_enheter,genomsnittligt_pris
0,Electronics,1364228.07,1753,789.320097
1,Sports,734653.42,1401,523.211181
2,Clothing,586744.91,1655,356.991120
3,Home,511070.87,1278,403.560959
4,Beauty,163053.97,659,250.573029
5,Toys,143015.99,717,201.582026


In [29]:
stad_analys = df.groupby("city").agg(
    total_intäkt=("revenue", "sum"),
    antal_sålda_enheter=("units", "sum"),
    genomsnittligt_order_värde=("revenue", "mean"),
    antal_order=("order_id", "nunique")
).reset_index()

stad_analys = stad_analys.sort_values(
    by="total_intäkt",
    ascending=False
)

stad_analys = stad_analys.reset_index(drop=True)

display(stad_analys)

,city,total_intäkt,antal_sålda_enheter,genomsnittligt_order_värde,antal_order
0,Stockholm,1227201.94,2619,1365.074461,899
1,Göteborg,859019.65,1788,1465.903840,586
2,Malmö,567926.32,1211,1434.157374,396
3,Uppsala,551478.42,1165,1447.449921,381
4,Västerås,297140.90,680,1248.491176,238


In [30]:
stad_och_kategori = df.groupby(
    ["city", "category"]
).agg(
    total_intäkt=("revenue", "sum")
).reset_index()

stad_och_kategori = stad_och_kategori.sort_values(
    by=["city", "total_intäkt"],
    ascending=[True, False]
)

stad_och_kategori = stad_och_kategori.reset_index(drop=True)

display(stad_och_kategori)

,city,category,total_intäkt
0,Göteborg,Electronics,340629.30
1,Göteborg,Sports,173114.39
2,Göteborg,Home,137851.89
3,Göteborg,Clothing,128566.21
4,Göteborg,Beauty,42693.20
5,Göteborg,Toys,36164.66
6,Malmö,Electronics,234714.72
7,Malmö,Sports,123529.32
8,Malmö,Home,89276.45
9,Malmö,Clothing,77554.64


Månadsvis försäljningsanalys

I detta avsnitt undersöker vi hur försäljningsintäkterna förändras mellan månaderna under perioden januari till juni 2024.

In [31]:
månads_intäkt = revenue_by_month(df)
display(månads_intäkt)

,month,total_intäkt
0,2024-01-01,664083.32
1,2024-02-01,548462.03
2,2024-03-01,487602.00
3,2024-04-01,580194.83
4,2024-05-01,632165.16
5,2024-06-01,590259.89
